# NLP Classifiers: Language Identification & Sentiment Analysis

Three models, trained from scratch every run and each saved as a `.pkl` you can
download from Colab and keep using locally:

1. **Setup** — imports and config
2. **Language Classifier** (TF-IDF + Naive Bayes, scikit-learn) → `lang_classifier.pkl`
3. **Language Classifier — Evaluation** — accuracy, macro F1, classification report
4. **Language Classifier — Manual Testing**
5. **Sentiment Bi-LSTM** (pure PyTorch) → `bilstm_sentiment.pkl`
6. **Sentiment Transformer fine-tune** (DistilRoBERTa, PyTorch/HF `Trainer`) → `roberta_sentiment.pkl`
7. **Combined Test** — language detection + sentiment on a sample query

All three model sections are self-contained and end with a pickle export, so you
can pull any of them down individually (Colab file browser, or `files.download(...)`)
and continue on your local machine.


## 1. Setup

In [1]:
# Core imports used throughout the notebook
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.model_selection import train_test_split

from datasets import load_dataset

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

TARGET_LANGS = ["en", "es", "fr", "de"]


Using device: cuda


## 2. Language Classifier (train from scratch)

TF-IDF (character n-grams) + Naive Bayes on `papluca/language-identification`.
Trained fresh every run, then pickled directly with `pickle` (a scikit-learn
`Pipeline` is a plain Python object, so this is the natural format for it).

In [2]:
print("Loading papluca/language-identification dataset...")
lang_dataset = load_dataset("papluca/language-identification")
lang_train_df = pd.DataFrame(lang_dataset["train"])
lang_test_df = pd.DataFrame(lang_dataset["test"])

lang_train_filt = lang_train_df[lang_train_df["labels"].isin(TARGET_LANGS)]
lang_test_filt = lang_test_df[lang_test_df["labels"].isin(TARGET_LANGS)]

lang_clf = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(1, 3))),
    ("clf", MultinomialNB())
])

print("Training language classifier...")
lang_clf.fit(lang_train_filt["text"], lang_train_filt["labels"])

LANG_PKL_PATH = "lang_classifier.pkl"
with open(LANG_PKL_PATH, "wb") as f:
    pickle.dump(lang_clf, f)
print(f"Saved trained classifier to '{LANG_PKL_PATH}'.")


Loading papluca/language-identification dataset...


README.md:   0%|          | 0.00/4.99k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 12.0MB            

train.csv: downloading bytes:           |  0.00B            

valid.csv:   0%|          | 0.00/1.71M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.69M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Training language classifier...
Saved trained classifier to 'lang_classifier.pkl'.


## 3. Language Classifier — Evaluation

In [3]:
y_true = lang_test_filt["labels"]
y_pred = lang_clf.predict(lang_test_filt["text"])

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("--- Language Model Test Metrics ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Macro F1:  {macro_f1:.4f}\n")
print(classification_report(y_true, y_pred))


--- Language Model Test Metrics ---
Accuracy:  1.0000
Macro F1:  1.0000

              precision    recall  f1-score   support

          de       1.00      1.00      1.00       500
          en       1.00      1.00      1.00       500
          es       1.00      1.00      1.00       500
          fr       1.00      1.00      1.00       500

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



## 4. Language Classifier — Manual Testing

Quick sanity check on hand-written sentences in each target language.

In [4]:
test_sentences = [
    ("Hello, how are you doing today?", "en"),
    ("Hola, ¿cómo estás hoy?", "es"),
    ("Bonjour, comment ça va aujourd'hui ?", "fr"),
    ("Hallo, wie geht es dir heute?", "de"),
]

results = []
for text, expected in test_sentences:
    pred = lang_clf.predict([text])[0]
    results.append({
        "text": text,
        "expected": expected,
        "predicted": pred,
        "correct": pred == expected
    })

results_df = pd.DataFrame(results)
display(results_df)
print(f"\nManual test accuracy: {results_df['correct'].mean():.2%}")


,text,expected,predicted,correct
0,"Hello, how are you doing today?",en,en,True
1,"Hola, ¿cómo estás hoy?",es,es,True
2,"Bonjour, comment ça va aujourd'hui ?",fr,fr,True
3,"Hallo, wie geht es dir heute?",de,de,True



Manual test accuracy: 100.00%


## 5. Sentiment Analysis — Bi-LSTM (PyTorch)

Trained from scratch on `dair-ai/emotion` (6-way emotion classes), rewritten in
pure PyTorch (no Keras/TensorFlow) so the vocabulary and weights are plain
Python objects — easy to pickle and easy to load back into a PyTorch `nn.Module`
locally with no TF dependency at all. Macro F1 is computed on the validation
split after every epoch.

In [5]:
emotion_ds = load_dataset("dair-ai/emotion")
df_train = pd.DataFrame(emotion_ds["train"])

print("--- Data Preview (dair-ai/emotion) ---")
label_names = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
df_train["label_name"] = df_train["label"].map(label_names)
display(df_train.head())
print("\nLabel Distribution:")
print(df_train["label_name"].value_counts())


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

--- Data Preview (dair-ai/emotion) ---


,text,label,label_name
0,i didnt feel humiliated,0,sadness
1,i can go from feeling so hopeless to so damned...,0,sadness
2,im grabbing a minute to post i feel greedy wrong,3,anger
3,i am ever feeling nostalgic about the fireplac...,2,love
4,i am feeling grouchy,3,anger



Label Distribution:
label_name
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64


In [6]:
# --- Build a simple word-level vocabulary ---
MAX_VOCAB = 10000
MAX_LEN = 100
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"

def simple_tokenize(text):
    return text.lower().split()

word_counts = {}
for text in df_train["text"]:
    for tok in simple_tokenize(text):
        word_counts[tok] = word_counts.get(tok, 0) + 1

most_common = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:MAX_VOCAB - 2]
vocab = {PAD_TOKEN: 0, OOV_TOKEN: 1}
for word, _ in most_common:
    vocab[word] = len(vocab)

def encode(text, vocab, max_len=MAX_LEN):
    ids = [vocab.get(tok, vocab[OOV_TOKEN]) for tok in simple_tokenize(text)][:max_len]
    ids = ids + [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

print(f"Vocabulary size: {len(vocab)}")


Vocabulary size: 10000


In [7]:
# --- Dataset / DataLoader ---
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx], self.vocab, self.max_len)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

texts = df_train["text"].tolist()
labels = df_train["label"].tolist()
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, random_state=42
)

train_dataset = EmotionDataset(train_texts, train_labels, vocab)
val_dataset = EmotionDataset(val_texts, val_labels, vocab)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [8]:
# --- Model definition ---
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=64, num_classes=6, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm1 = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(hidden_dim * 2, hidden_dim // 2, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.lstm1(emb)
        out, (h_n, _) = self.lstm2(out)
        # concat final forward/backward hidden states
        h_final = torch.cat((h_n[-2], h_n[-1]), dim=1)
        x = self.relu(self.fc1(h_final))
        x = self.dropout(x)
        return self.fc2(x)

bilstm_model = BiLSTMClassifier(vocab_size=len(vocab)).to(device)
optimizer = torch.optim.Adam(bilstm_model.parameters())
criterion = nn.CrossEntropyLoss()
print(bilstm_model)


BiLSTMClassifier(
  (embedding): Embedding(10000, 64, padding_idx=0)
  (lstm1): LSTM(64, 64, batch_first=True, bidirectional=True)
  (lstm2): LSTM(128, 32, batch_first=True, bidirectional=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=64, out_features=6, bias=True)
  (relu): ReLU()
)


In [9]:
# --- Training loop with macro F1 each epoch ---
EPOCHS = 3

def evaluate(model, loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(y_batch.numpy())
    return np.array(all_targets), np.array(all_preds)

print("Starting Bi-LSTM Training...")
for epoch in range(EPOCHS):
    bilstm_model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = bilstm_model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)

    avg_loss = total_loss / len(train_dataset)
    y_val_true, y_val_pred = evaluate(bilstm_model, val_loader)
    val_acc = accuracy_score(y_val_true, y_val_pred)
    val_macro_f1 = f1_score(y_val_true, y_val_pred, average="macro")
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss: {avg_loss:.4f} - val_accuracy: {val_acc:.4f} - val_macro_f1: {val_macro_f1:.4f}")


Starting Bi-LSTM Training...
Epoch 1/3 - loss: 1.5827 - val_accuracy: 0.3350 - val_macro_f1: 0.1208
Epoch 2/3 - loss: 1.3363 - val_accuracy: 0.5669 - val_macro_f1: 0.3706
Epoch 3/3 - loss: 0.8977 - val_accuracy: 0.6769 - val_macro_f1: 0.4650


In [10]:
# --- Final evaluation summary ---
y_val_true, y_val_pred = evaluate(bilstm_model, val_loader)
print("--- Bi-LSTM Final Validation Metrics ---")
print(f"Macro F1: {f1_score(y_val_true, y_val_pred, average='macro'):.4f}\n")
print(classification_report(y_val_true, y_val_pred, target_names=list(label_names.values())))


--- Bi-LSTM Final Validation Metrics ---
Macro F1: 0.4650

              precision    recall  f1-score   support

     sadness       0.88      0.77      0.82       494
         joy       0.62      0.97      0.75       503
        love       0.32      0.08      0.12       159
       anger       0.64      0.35      0.45       217
        fear       0.58      0.72      0.64       174
    surprise       0.00      0.00      0.00        53

    accuracy                           0.68      1600
   macro avg       0.50      0.48      0.47      1600
weighted avg       0.65      0.68      0.63      1600



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
# --- Export: model weights + vocab, all in one pickle ---
BILSTM_PKL_PATH = "bilstm_sentiment.pkl"

bilstm_export = {
    "state_dict": bilstm_model.state_dict(),
    "vocab": vocab,
    "max_len": MAX_LEN,
    "label_names": label_names,
    "model_config": {"embed_dim": 64, "hidden_dim": 64, "num_classes": 6}
}

# torch.save uses pickle under the hood and handles tensors correctly,
# so the .pkl extension here is a real pickle file you can also
# `pickle.load()` directly if you prefer.
torch.save(bilstm_export, BILSTM_PKL_PATH)
print(f"Saved Bi-LSTM weights + vocab to '{BILSTM_PKL_PATH}'.")

# To reload locally:
#   ckpt = torch.load('bilstm_sentiment.pkl', map_location='cpu')
#   model = BiLSTMClassifier(vocab_size=len(ckpt['vocab']))
#   model.load_state_dict(ckpt['state_dict'])


Saved Bi-LSTM weights + vocab to 'bilstm_sentiment.pkl'.


## 6. Sentiment Analysis — Transformer Fine-Tune (DistilRoBERTa, PyTorch)

Fine-tunes `distilroberta-base` on a 3-class sentiment remap of `dair-ai/emotion`
(negative / neutral / positive) using the Hugging Face `Trainer`, which runs on
PyTorch under the hood. `compute_metrics` reports **macro F1** every eval epoch.
Trained from scratch each run, then the state dict + config are pickled for
easy local extraction.

In [12]:
!pip install -q transformers accelerate


In [13]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

MODEL_NAME = "distilroberta-base"

# 1. Load dataset and remap the 6 emotion labels to 3 sentiment classes
sentiment_ds = load_dataset("dair-ai/emotion")
emotion_to_sentiment = {0: 0, 1: 2, 2: 2, 3: 0, 4: 0, 5: 1}  # neg, joy->pos, love->pos, anger->neg, fear->neg, surprise->neutral

def remap_labels(example):
    example["label"] = emotion_to_sentiment[example["label"]]
    return example

sentiment_ds = sentiment_ds.map(remap_labels)

# 2. Tokenization
roberta_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return roberta_tokenizer(examples["text"], truncation=True)

tokenized_datasets = sentiment_ds.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=roberta_tokenizer)

# 3. Load model
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2}
)

# 4. Macro F1 during evaluation
def compute_metrics(eval_pred):
    logits, targets = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(targets, preds),
        "macro_f1": f1_score(targets, preds, average="macro")
    }

# 5. Training
training_args = TrainingArguments(
    output_dir="./roberta_sentiment_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=roberta_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.119965,0.075157,0.974000,0.936023
2,0.063039,0.070991,0.978000,0.938598
3,0.032693,0.069746,0.981000,0.944843


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1500, training_loss=0.10522306601206462, metrics={'train_runtime': 150.097, 'train_samples_per_second': 319.793, 'train_steps_per_second': 9.994, 'total_flos': 642577693287168.0, 'train_loss': 0.10522306601206462, 'epoch': 3.0})

In [14]:
# Final macro F1 on the held-out test split
test_results = trainer.evaluate(tokenized_datasets["test"])
print("--- DistilRoBERTa Test Metrics ---")
print(f"Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"Macro F1:  {test_results['eval_macro_f1']:.4f}")


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.032693,0.073699,3,0.975500,0.901660


--- DistilRoBERTa Test Metrics ---
Accuracy:  0.9755
Macro F1:  0.9017


In [15]:
# --- Export: state dict + tokenizer + config, all in one pickle ---
ROBERTA_PKL_PATH = "roberta_sentiment.pkl"

roberta_export = {
    "state_dict": roberta_model.state_dict(),
    "config": roberta_model.config,
    "model_name": MODEL_NAME,
    "id2label": {0: "negative", 1: "neutral", 2: "positive"},
}

torch.save(roberta_export, ROBERTA_PKL_PATH)
print(f"Saved DistilRoBERTa weights + config to '{ROBERTA_PKL_PATH}'.")

# Tokenizer files aren't picklable as a single tensor blob (they're a folder
# of vocab/merge files), so save those alongside for local reuse:
roberta_tokenizer.save_pretrained("./roberta_tokenizer")
print("Tokenizer saved to './roberta_tokenizer' (zip/download this folder too).")

# To reload locally:
#   ckpt = torch.load('roberta_sentiment.pkl', map_location='cpu')
#   model = AutoModelForSequenceClassification.from_pretrained(
#       ckpt['model_name'], config=ckpt['config']
#   )
#   model.load_state_dict(ckpt['state_dict'])
#   tokenizer = AutoTokenizer.from_pretrained('./roberta_tokenizer')


Saved DistilRoBERTa weights + config to 'roberta_sentiment.pkl'.
Tokenizer saved to './roberta_tokenizer' (zip/download this folder too).


## 7. Combined Test

Runs a query through both the language classifier and the fine-tuned
sentiment model (using the in-memory models trained above).

In [17]:
def test_chatbot_components(user_query):
    # 1. Detect language (TF-IDF / Naive Bayes pipeline)
    detected_lang = lang_clf.predict([user_query])[0]

    # 2. Detect sentiment (fine-tuned transformer)
    inputs = roberta_tokenizer(user_query, return_tensors="pt", truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()} # Move inputs to the same device as the model
    roberta_model.eval()
    with torch.no_grad():
        logits = roberta_model(**inputs).logits
        prediction = torch.argmax(logits, dim=-1).item()
    sentiment_label = roberta_model.config.id2label[prediction]

    print("--- Chatbot Analysis ---")
    print(f"User Query: \"{user_query}\"")
    print(f"Detected Language:  {detected_lang.upper()}")
    print(f"Detected Sentiment: {sentiment_label.upper()}\n")

test_chatbot_components("I am so happy that my order arrived early, thank you!")
test_chatbot_components("¡Estoy muy enfadado porque mi pedido no ha llegado!")

--- Chatbot Analysis ---
User Query: "I am so happy that my order arrived early, thank you!"
Detected Language:  EN
Detected Sentiment: POSITIVE

--- Chatbot Analysis ---
User Query: "¡Estoy muy enfadado porque mi pedido no ha llegado!"
Detected Language:  ES
Detected Sentiment: NEGATIVE

